# Animation: KNN Decision Boundary vs. k

Watch the decision boundary change as `k` increases from 1 to 30:

- **Small k (e.g. k=1):** extremely jagged boundary → **overfitting**
- **Medium k (e.g. k=7–15):** smooth, reasonable boundary → **good generalization**
- **Large k (e.g. k=30):** too smooth, ignores structure → **underfitting**

This is the **bias-variance tradeoff** made visible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from IPython.display import HTML
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# ── Data: Iris, 2 features, 3 classes ───────────────────────────
iris = load_iris()
X = iris.data[:, [2, 3]]   # petal length, petal width
y = iris.target

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# ── Mesh grid for decision boundary ─────────────────────────────
h = 0.02
x_min, x_max = X_sc[:, 0].min() - 0.5, X_sc[:, 0].max() + 0.5
y_min, y_max = X_sc[:, 1].min() - 0.5, X_sc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))

# Pre-compute boundaries for every k
k_values = list(range(1, 31))
Z_all = {}
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_sc, y)
    Z_all[k] = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

print('Precomputed boundaries for k =', k_values)

In [ ]:
# ── Build animation ─────────────────────────────────────────────
CMAP_BG   = ListedColormap(['#fadbd8', '#d6eaf8', '#d5f5e3'])
COLORS_PT = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('Petal Length (scaled)', fontsize=11)
ax.set_ylabel('Petal Width (scaled)', fontsize=11)

# Plot points once (they don't move)
for cls, color, name in zip([0, 1, 2], COLORS_PT, iris.target_names):
    mask = y == cls
    ax.scatter(X_sc[mask, 0], X_sc[mask, 1],
               color=color, edgecolors='white', linewidths=0.5,
               s=55, zorder=3, label=name)
ax.legend(loc='upper left', fontsize=9)

# Placeholders
mesh_im = [None]
title_obj = ax.set_title('', fontsize=13)

# Bias-variance label
bv_text = ax.text(0.98, 0.03, '', transform=ax.transAxes,
                  fontsize=11, ha='right', va='bottom',
                  bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

def update(frame):
    k = k_values[frame]
    if mesh_im[0] is not None:
        mesh_im[0].remove()
    mesh_im[0] = ax.pcolormesh(xx, yy, Z_all[k], cmap=CMAP_BG, alpha=0.55, zorder=1)
    title_obj.set_text(f'KNN Decision Boundary  (k = {k})')

    if k <= 3:
        label = 'Overfitting (high variance)'
        color = '#e74c3c'
    elif k >= 20:
        label = 'Underfitting (high bias)'
        color = '#e67e22'
    else:
        label = 'Good balance'
        color = '#27ae60'

    bv_text.set_text(label)
    bv_text.get_bbox_patch().set_facecolor(color)
    bv_text.set_color('white')
    return mesh_im[0], title_obj, bv_text

anim = FuncAnimation(fig, update, frames=len(k_values),
                     interval=300, repeat=True, blit=False)
plt.tight_layout()
plt.close()
HTML(anim.to_jshtml())

## What to notice

| k | Boundary | Tendency |
|---|----------|----------|
| 1 | Extremely jagged — memorizes every point | Overfit |
| 5–10 | Smooth but responsive | Usually best |
| 20+ | Very smooth, loses fine-grained class boundaries | Underfit |

**The bias-variance tradeoff:** Low k → low bias, high variance. High k → high bias, low variance.

Cross-validation picks the k that generalizes best to unseen data.